In [1]:
from pathlib import Path
import json
import pandas as pd

rows = []

for file in Path("archives").rglob("*.json"):
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for trial in data["resultats"]:
        if trial.get("task") != "target":
            continue

        participant_id = trial["participant_id"]
        rt = trial["rt"]
        if not rt:
            continue

        valence = trial["valence"]
        attr = trial["baseline_attrac"]
        response = trial["response_meaning"]

        # Congruence
        if (valence == "positif" and attr == "Attractif") or \
        (valence == "negatif" and attr == "Non attractif"):
            congruence = "congruent"
        else:
            congruence = "incongruent"
        if valence=='neutre':
            congruence='neutre'

        # Résultat
        if attr == "Attractif" and response == "Attractif":
            result = 1
        elif attr == "Unattractif" and response == "Non attractif":
            result = 1
        else:
            result = 0

        rows.append({
            "id": participant_id,
            "valence_mot":valence,
            "actracttiveness_face":attr, 
            "congruence": congruence,
            "correct": result,
            "rt": rt/1000
        })

# Transformer en DataFrame
df = pd.DataFrame(rows)

# Export en CSV
df.to_csv("resultats.csv", index=False, encoding="utf-8")

print("Fichier CSV généré")

Fichier CSV généré


### Condition controle

Visage précédé d'un mot neutre. En théorie, 100% de précision:

In [2]:
control_csv=df[df['congruence']=='neutre']
df.to_csv("resultats_condition_neutre.csv", index=False, encoding="utf-8")
control_csv

,id,valence_mot,actracttiveness_face,congruence,correct,rt
3,9at0mps0,neutre,Unattractif,neutre,0,0.185
5,9at0mps0,neutre,Attractif,neutre,1,0.092
7,9at0mps0,neutre,Unattractif,neutre,0,0.261
8,9at0mps0,neutre,Attractif,neutre,1,0.120
11,9at0mps0,neutre,Unattractif,neutre,0,0.097
...,...,...,...,...,...,...
2703,6ipwu4mk,neutre,Attractif,neutre,0,0.869
2704,6ipwu4mk,neutre,Unattractif,neutre,1,0.603
2705,6ipwu4mk,neutre,Unattractif,neutre,1,0.843
2719,6ipwu4mk,neutre,Unattractif,neutre,1,0.620


In [3]:
result = (
    control_csv.groupby('id')['correct']
    .mean()
    .reset_index(name='pourcentage_correct')
)

result['pourcentage_correct'] *= 100
result[result['pourcentage_correct']>70].count()

id                     31
pourcentage_correct    31
dtype: int64

### Condition positive

In [4]:
positive_csv=df[df['valence_mot']=='positif']
positive_csv.to_csv("resultats_condition_positive.csv", index=False, encoding="utf-8")
positive_csv

,id,valence_mot,actracttiveness_face,congruence,correct,rt
0,9at0mps0,positif,Attractif,congruent,0,0.223
1,9at0mps0,positif,Unattractif,incongruent,0,0.295
4,9at0mps0,positif,Unattractif,incongruent,0,1.067
10,9at0mps0,positif,Unattractif,incongruent,0,0.455
12,9at0mps0,positif,Unattractif,incongruent,0,0.407
...,...,...,...,...,...,...
2712,6ipwu4mk,positif,Attractif,congruent,0,0.623
2713,6ipwu4mk,positif,Unattractif,incongruent,1,0.500
2716,6ipwu4mk,positif,Attractif,congruent,0,0.733
2717,6ipwu4mk,positif,Unattractif,incongruent,1,0.478


In [5]:
result_p = (
    positive_csv.groupby('id')['correct']
    .mean()
    .reset_index(name='pourcentage_correct')
)

result_p['pourcentage_correct'] *= 100
result_p['pourcentage_correct'].mean

<bound method Series.mean of 0      89.473684
1      65.000000
2     100.000000
3      65.000000
4      61.538462
5      65.000000
6      75.000000
7      84.210526
8      45.000000
9      60.000000
10     63.157895
11     75.000000
12    100.000000
13     80.000000
14     90.000000
15     88.888889
16     70.000000
17     80.000000
18     63.157895
19     95.000000
20     68.421053
21     50.000000
22     95.000000
23     82.352941
24     95.000000
25    100.000000
26     89.473684
27     80.000000
28     60.000000
29     85.000000
30     84.210526
31     60.000000
32     55.000000
33     65.000000
34     78.947368
35     73.684211
36     90.000000
37     73.684211
38     83.333333
39     90.000000
40     57.894737
41     90.000000
42     77.777778
43     78.947368
44     55.000000
45     65.000000
46     65.000000
Name: pourcentage_correct, dtype: float64>

### Condition neagtive


In [6]:
negative_csv=df[df['valence_mot']=='negatif']
negative_csv.to_csv("resultats_condition_negatif.csv", index=False, encoding="utf-8")
negative_csv

,id,valence_mot,actracttiveness_face,congruence,correct,rt
2,9at0mps0,negatif,Unattractif,incongruent,1,0.109
6,9at0mps0,negatif,Attractif,incongruent,1,0.222
9,9at0mps0,negatif,Attractif,incongruent,1,0.518
17,9at0mps0,negatif,Unattractif,incongruent,0,0.195
21,9at0mps0,negatif,Attractif,incongruent,1,0.502
...,...,...,...,...,...,...
2714,6ipwu4mk,negatif,Unattractif,incongruent,1,0.640
2715,6ipwu4mk,negatif,Unattractif,incongruent,1,0.629
2720,6ipwu4mk,negatif,Attractif,incongruent,0,0.879
2721,6ipwu4mk,negatif,Attractif,incongruent,1,0.546


In [7]:
result_n = (
    negative_csv.groupby('id')['correct']
    .mean()
    .reset_index(name='pourcentage_correct')
)

result_n['pourcentage_correct'] *= 100
result_n['pourcentage_correct'].mean

<bound method Series.mean of 0      89.473684
1      70.000000
2      85.000000
3      78.947368
4      70.000000
5      60.000000
6      75.000000
7     100.000000
8      55.000000
9      70.000000
10     89.473684
11     77.777778
12     95.000000
13     47.368421
14     80.000000
15     92.857143
16     70.000000
17     85.000000
18     55.000000
19    100.000000
20     66.666667
21     52.631579
22     94.444444
23     66.666667
24     90.000000
25     90.000000
26     70.000000
27     85.000000
28     75.000000
29     63.157895
30     89.473684
31     75.000000
32     63.157895
33     77.777778
34    100.000000
35     63.157895
36     94.736842
37     61.111111
38     90.000000
39     85.000000
40     57.894737
41     89.473684
42     87.500000
43     85.000000
44     65.000000
45     60.000000
46     85.000000
Name: pourcentage_correct, dtype: float64>